In [ ]:
import cv2
import PIL
import numpy as np
import matplotlib.pyplot as plt

print("NumPy Version: ", np.__version__)
print("OpenCV Version: ", cv2.__version__)
print("PIL Version: ", PIL.__version__)

In [ ]:
import time

# Since I am using WSL, it needs an address like this to access Windows cameras via ffmpeg/ffplay
cap = cv2.VideoCapture("udp://0.0.0.0:5000")
images = []
for i in range(10):
    ret, frame = cap.read()
    print(f"Frame {i+1}: Read Status: {ret}, Frame Shape: {frame.shape if ret else 'N/A'}")
    if ret:
        images.append(frame)
    time.sleep(1)

cap.release()
print("Captured Images: ", len(images))

# Display the captured images
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, ax in enumerate(axes.flatten()):
    if i < len(images):
        ax.imshow(cv2.cvtColor(images[i], cv2.COLOR_BGR2RGB))
        ax.set_title(f"Image {i+1}")
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# convert images to gray scale
gray_images = []
for i, img in enumerate(images):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray_images.append(gray)
    print(f"Image {i+1}: Converted to Grayscale, Shape: {gray.shape}")

# Display the grayscale images
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, ax in enumerate(axes.flatten()):
    if i < len(gray_images):
        ax.imshow(gray_images[i], cmap='gray')
        ax.set_title(f"Gray Image {i+1}")
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Convert lists to numpy arrays for easier processing
images_array = np.array(images)
gray_images_array = np.array(gray_images)

# Compute the sum of absolute differences between consecutive images
diff_image = np.zeros_like(images_array[0], dtype=np.float32)
diff_gray_images = np.zeros_like(gray_images_array[0], dtype=np.float32)
for i in range(1, len(gray_images_array)):
    diff_image += np.abs(
        images_array[i-1].astype(np.float32) - images_array[i].astype(np.float32)
    )
    diff_gray_images += np.abs(
        gray_images_array[i-1].astype(np.float32) - gray_images_array[i].astype(np.float32)
    )

# Clip the differences to the valid range [0, 255] and convert back to uint8
diff_image = np.clip(diff_image, 0, 255).astype(np.uint8)
diff_gray_images = np.clip(diff_gray_images, 0, 255).astype(np.uint8)

# Compute the mean and standard deviation of the images
mean_diff_images = np.mean(images_array, axis=0).astype(np.uint8)
std_diff_images = np.std(images_array, axis=0).astype(np.uint8)

# Compute the mean and standard deviation of the grayscale images
mean_diff_gray_images = np.mean(gray_images_array, axis=0).astype(np.uint8)
std_diff_gray_images = np.std(gray_images_array, axis=0).astype(np.uint8)

print("Mean Diff Image Shape: ", mean_diff_images.shape)
print("Std Diff Image Shape: ", std_diff_images.shape)
print("Mean Diff Gray Image Shape: ", mean_diff_gray_images.shape)
print("Std Diff Gray Image Shape: ", std_diff_gray_images.shape)

# Display the results
fig, axes = plt.subplots(3, 2, figsize=(12, 15))
axes[0, 0].imshow(diff_image)
axes[0, 0].set_title("Sum of Absolute Differences (Color)")
axes[0, 0].axis('off')
axes[0, 1].imshow(diff_gray_images, cmap='gray')
axes[0, 1].set_title("Sum of Absolute Differences (Grayscale)")
axes[0, 1].axis('off')
axes[1, 0].imshow(mean_diff_images)
axes[1, 0].set_title("Mean Image (Color)")
axes[1, 0].axis('off')
axes[1, 1].imshow(mean_diff_gray_images, cmap='gray')
axes[1, 1].set_title("Mean Image (Grayscale)")
axes[1, 1].axis('off')
axes[2, 0].imshow(std_diff_images)
axes[2, 0].set_title("Standard Deviation Image (Color)")
axes[2, 0].axis('off')
axes[2, 1].imshow(std_diff_gray_images, cmap='gray')
axes[2, 1].set_title("Standard Deviation Image (Grayscale)")
axes[2, 1].axis('off')
plt.tight_layout()
plt.show()